<a href="https://colab.research.google.com/github/hiroaki-com/colab-ollama-server/blob/main/ollama_colab_free_server_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Google ColabのGPU上でOllamaを動作させ、ngrokトンネル経由でOllamaエンドポイントを公開するLLMサーバー。

主な特徴
- ✅ 完全無料。外部APIへのデータ送信なし、ローカル推論によるプライバシー保護
- 🎛️ モデルをUIで選択し、プルから起動まで自動実行
- 🚀 ContinueやClaude Codeから即座に接続可能


> ※ 接続先ツールの`ベースURL` 末尾に `/v1` を追加することで、OpenAI 互換クライアント（Codex CLI 等）からの利用も想定しています。 例）`https://xxxx.ngrok-free.app/v1`

In [ ]:
#@title 📋 Model Registry

# @markdown ### モデル設定
# @markdown > 追加したいモデル名をカンマ区切りで入力して下さい。 目安:`8B`推奨・`14B`可・`20B+`低速
# @markdown > - 正式なモデル名はこちらから：https://ollama.com/search

model_list = "qwen3:8b, qwen3:14b, qwen2.5-coder:7b, qwen2.5-coder:14b, ministral-3:8b, ministral-3:14b, devstral-small-2:24b, gpt-oss:20b, deepseek-r1:8b, glm-4.7-flash, qwen3.5:9b, qwen3.5:4b, qwen3.5:2b, qwen3.5:0.8b" #@param {type:"string"}

# @markdown ### context length (num_ctx)
# @markdown > T4 (16GB VRAM) 目安
# @markdown > | モデルサイズ | 推奨 ctx | 備考 |
# @markdown > |---|---|---|
# @markdown > | ~4B  | 32768 | 余裕あり |
# @markdown > | ~8B  | 16384 | 標準推奨 |
# @markdown > | ~14B | 8192  | VRAMタイト |
# @markdown > | ~20B+ | 4096 | 最小運用 |
# @markdown >
# @markdown > デフォルト(0)は Ollama 任せ（= 4096 相当）。
num_ctx = 16384 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ モデルリストが空です。少なくとも1つのモデルを入力してください。")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 モデルを選択</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    '起動するモデルを1つ選択し、次のセルを実行してください。</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(padding='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

print(f"✅ モデルリストを読み込みました: {len(AVAILABLE_MODELS)}個")
print("➡️ モデルを選択後、次のセル（Server）を実行してください。")


In [ ]:
#@title 🚀 Ollama Colab Free Server

# @markdown > 初回は下記URLより`Ngrok`で無料アカウントを作成して、取得したトークンを貼り付けてください。
# @markdown > - https://dashboard.ngrok.com/get-started/your-authtoken

ngrok_token = "" #@param {type:"string"}

MAX_HEALTH_RETRIES   = 150
HEALTH_CHECK_TIMEOUT = 2
STATUS_POLL_INTERVAL = 30

BLUE  = "\033[34m"
GREEN = "\033[32m"
GRAY  = "\033[90m"
RESET = "\033[0m"

selected_model = model_selector.value
effective_ctx  = num_ctx if num_ctx > 0 else None  # None = Ollama デフォルト

print(f"\nOLLAMA COLAB SERVER 🚀")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"  {BLUE}◦{RESET} System   依存パッケージのインストール (zstd)")
!which zstd > /dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)

print(f"  {BLUE}◦{RESET} System   Ollama のインストール")
!which ollama > /dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print(f"  {BLUE}◦{RESET} System   pyngrok のインストール")
!python -c "import pyngrok" > /dev/null 2>&1 || pip install -q pyngrok

import re
import subprocess
import time
import os
import requests

# シェルインジェクション防止のためモデル名を事前検証
if not re.fullmatch(r'[a-zA-Z0-9._:/-]+', selected_model):
    raise ValueError(f"モデル名が不正です: {selected_model}")

os.environ['OLLAMA_HOST']              = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE']        = '24h'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_FLASH_ATTENTION']   = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']     = 'q8_0'

process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(MAX_HEALTH_RETRIES):
    try:
        if requests.get("http://0.0.0.0:11434/api/tags", timeout=HEALTH_CHECK_TIMEOUT).status_code == 200:
            print(f"  {GREEN}✓{RESET} Ready    Ollama サーバー起動完了 (0.0.0.0:11434)")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.2)
else:
    raise RuntimeError("⚠️ Ollama サーバーの起動確認に失敗しました。")

print(f"  {BLUE}◦{RESET} Network  ngrok トンネル確立中 (port: 11434)")
from pyngrok import ngrok

if not ngrok_token:
    raise ValueError("⚠️ ngrok トークンが設定されていません。")

ngrok.set_auth_token(ngrok_token)
ngrok.kill()  # 前セッションのリモートエンドポイントも含めて解放
tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print(f"  {GREEN}✓{RESET} Public   {public_url}")

print(f"\n  ▶ Model    {selected_model}")
ctx_label = str(effective_ctx) if effective_ctx else "Ollama デフォルト (4096)"
print(f"  {GRAY}  num_ctx  {ctx_label}{RESET}")
print(f"  {GRAY}└ Pulling  ダウンロード開始... (初回: 5〜15分程度){RESET}")
subprocess.run(
    ["/usr/local/bin/ollama", "pull", selected_model],
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'},
    check=True
)
print(f"  {GREEN}✓{RESET} Loaded   モデルロード完了")

# warmup: num_ctx を options で指定してモデルをそのcontext lengthでVRAMにロードする
# keep_alive未指定でOLLAMA_KEEP_ALIVEが有効になる
warmup_payload = {
    "model": selected_model,
    "prompt": "hi",
    "stream": False,
}
if effective_ctx:
    warmup_payload["options"] = {"num_ctx": effective_ctx}

try:
    requests.post(
        "http://0.0.0.0:11434/api/generate",
        json=warmup_payload,
        timeout=180
    )
    ctx_info = f" (num_ctx: {effective_ctx})" if effective_ctx else ""
    print(f"  {GREEN}✓{RESET} Warmed   モデルをVRAMにロード済み{ctx_info}")
except requests.exceptions.RequestException as e:
    print(f"  ⚠️  Warmup   ウォームアップ失敗: {e}")

ctx_disp = str(effective_ctx) if effective_ctx else "4096 (Ollama デフォルト)"
print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"ENDPOINT : {public_url}")
print(f"NUM_CTX  : {ctx_disp}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"Continue Extension Config (~/.continue/config.yaml)")
print(f"{GRAY}VS Code 等 の Continue 拡張機能でローカル LLM を使用する際の設定例。{RESET}")
ctx_yaml = effective_ctx if effective_ctx else 4096
print(f"""
models:
  - title: {selected_model}
    provider: ollama
    model: {selected_model}
    apiBase: {public_url}
    contextLength: {ctx_yaml}
""")

print(f"Claude Code Setup (shell env)")
print(f"{GRAY}API エンドポイントを本サーバーへ切り替えて claude コマンドを使用する環境変数の設定例。{RESET}")
print(f"""
export ANTHROPIC_BASE_URL={public_url}
export ANTHROPIC_API_KEY=dummy
claude --model {selected_model}
""")

print(f"Codex Setup (VS Code 等の Codex 拡張機能 / CLI)")
print(f"{GRAY}Codex 拡張機能・CLI でローカル LLM を使用する際の設定例。{RESET}")
print(f"""
{GRAY}~/.codex/config.toml{RESET}
model = \"{selected_model}\"

{GRAY}shell env{RESET}
export OPENAI_BASE_URL={public_url}/v1
export OPENAI_API_KEY=dummy
code .    {GRAY}# 拡張機能の起動時{RESET}
codex     {GRAY}# CLIの起動時{RESET}
""")

try:
    start_time = time.time()
    while True:
        elapsed_min = int((time.time() - start_time) / 60)
        print(f"\r  {GREEN}●{RESET} Running  稼働中: {elapsed_min}分経過 | {public_url}", end="")
        time.sleep(STATUS_POLL_INTERVAL)
except KeyboardInterrupt:
    print(f"\n\n  {GRAY}セッションを終了しました。{RESET}")
finally:
    ngrok.disconnect(public_url)
